# ReAct Multi-hop RAG with GoodMem

Adapted from [Chandula Senevirathna’s Agentic_RAG](https://github.com/ChandulaSenevirathna/Agentic_RAG); original [license notice](LICENSE.md) retained.

Run cells from top to bottom. Shared Python modules let the notebooks and CLI exercise the same tested implementation.

## 1. Configure the clients
Run `uv sync`, configure `.env`, and follow the README setup first. GoodMem handles document storage, chunking, embedding and retrieval; LangGraph controls the agent. Groq is the upstream chat default; Cohere is also supported.

In [ ]:
from IPython.display import Markdown, display
from langchain_core.messages import HumanMessage

from goodmem_rag.config import Settings, chat_model
from goodmem_rag.ingestion import setup
from goodmem_rag.retrieval import GoodMemRetriever, make_tools
from goodmem_rag.agents import answer_text, tool_calls
from goodmem_rag.evaluation import COMPARISON_QUESTION, SEQUENTIAL_QUESTION

settings = Settings.from_env()
llm = chat_model()

## 2. Ingest the two knowledge bases
The same six official pages as upstream are fetched in Markdown form, with an HTML fallback. Whole pages go into two GoodMem spaces. Server-side recursive chunking uses 1,000 characters with 200 characters of overlap. We wait for `COMPLETED`, reuse unchanged memories, and replace changed pages only after the new version has indexed. This cell can be rerun safely.

In [ ]:
state = setup(settings)
print({"spaces": state["spaces"], "documents": len(state["documents"]),
       "reranker_enabled": bool(state.get("reranker_id"))})

## 3. Inspect retrieval before asking the agent
`GoodMemRetriever` joins memory-definition events with chunk events to retain titles and source URLs. Stream warnings are surfaced as errors instead of silently looking like no matches. A configured reranker is applied inside GoodMem.

In [ ]:
client = settings.client()
retriever = GoodMemRetriever(client, state["spaces"]["langgraph"],
                             reranker_id=state.get("reranker_id"))
docs = retriever.invoke("What are state, nodes and edges in StateGraph?")
assert docs and all(d.metadata["source"] for d in docs)
for doc in docs[:2]:
    print(doc.metadata["source"], doc.metadata["score"])
    print(doc.page_content[:300], "\n")

## 4. Give the agent two tools
Each tool searches exactly one GoodMem space. The model chooses the tools and queries. Embedding credentials stay on the GoodMem server after registration; no local embedding model or vector database is loaded.

In [ ]:
tools = make_tools(client, state)
[(tool.name, tool.description) for tool in tools]

## 5. Build the agent
This keeps the upstream prebuilt `langchain.agents.create_agent` approach. The agent interprets results and decides whether another lookup is needed. Tool-call middleware enforces four total searches, two per tool, and an eight-model-call limit. See [agents.py](goodmem_rag/agents.py) for the builder. There are no separate grading or rewriting nodes.

In [ ]:
from goodmem_rag.agents import build_react_agent
agent = build_react_agent(llm, tools)

In [ ]:
display(Markdown("```mermaid\n" + agent.get_graph().draw_mermaid() + "\n```"))

## 6. A single-space question

In [ ]:
single = agent.invoke({"messages": [HumanMessage(content="What is a checkpointer used for in LangGraph? Cite the docs.")]},
                      {"recursion_limit": 60})
print(answer_text(single))
print(tool_calls(single))

## 7. Compare both knowledge bases
Inspect actual calls: the model may request both tools in one round. Two tools alone do not prove sequential multi-hop reasoning.

In [ ]:
comparison = agent.invoke({"messages": [HumanMessage(content=COMPARISON_QUESTION)]},
                          {"recursion_limit": 60})
print(answer_text(comparison))
for call in tool_calls(comparison):
    print(call["name"], call["args"])

## 8. A dependent follow-up lookup
First identify the framework recommended by the LangGraph overview, then find its agent constructor in that framework’s docs. The evaluation checks that this uses at least two retrieval rounds and cites evidence from both spaces.

In [ ]:
multi = agent.invoke({"messages": [HumanMessage(content=SEQUENTIAL_QUESTION)]},
                     {"recursion_limit": 60})
print(answer_text(multi))
for message in multi["messages"]:
    if getattr(message, "tool_calls", None):
        print("Retrieval round:", [(c["name"], c["args"]) for c in message.tool_calls])

## 9. A question that needs no retrieval

In [ ]:
direct = agent.invoke({"messages": [HumanMessage(content="What is the capital of France?")]},
                      {"recursion_limit": 60})
print(answer_text(direct))
assert not tool_calls(direct)

## 10. Close connections
The indexed documents persist in GoodMem. Run `uv run goodmem-rag evaluate` for machine-readable results, and see [the integration report](docs/goodmem-rough-edges.md) for findings and limits.

In [ ]:
client.close()